# DiningCode 리뷰 수집 품질 점검

## tl;dr

성심당 본점 상세 페이지에서 `평가 더보기`를 71회 누른 뒤 356개 리뷰 카드가 렌더링됐습니다. 페이지 요약은 364명으로 표시되어 8건 차이가 남습니다. 버튼은 사라졌지만 이 차이를 해소하지 못해 CSV와 체크포인트를 미완료 상태로 보존했습니다.

## Context & Methods

분석 단위는 방문자 평가 카드 1개이며 CSV는 카드마다 한 행을 둡니다. 입력은 `crawler/data/diningcode_playwright_national_sungsimdang_20260923.csv.partial`와 같은 이름의 체크포인트입니다. 리뷰 ID 중복, 이름·별점·작성일 누락, 본문 빈값, 본문 잘림 표시, 페이지 평가 수와 행 수 차이를 확인합니다.

### Key Assumptions

- 페이지 요약의 364는 식당 평가 전체 수이고, 각 렌더링 카드가 하나의 평가를 나타냅니다.
- 빈 본문은 평점만 남긴 평가일 수 있으므로 본문 누락과 잘림을 구분합니다.
- 차이 8건의 원인은 현재 확인된 카드 DOM만으로 특정하지 않습니다.
- 현재 카드에서 작성자 표시명은 보이지만 작성자 링크나 안정적인 사용자 ID는 보이지 않았습니다. 이 CSV의 `user_name`은 사용자 식별자 대신 쓸 수 없습니다.

## Data

수집은 2026-09-23에 headed Playwright로 수행했습니다. `div.latter-graph` 카드와 `a.more-btn-inline` 본문 펼침을 사용했고, robots.txt가 허용하는 경로만 요청했습니다.

## Results

아래 셀은 행 수, 리뷰 ID 고유성, 필수 필드 누락, 본문 비어 있음과 평가 수 대조를 다시 계산합니다.

## Takeaways

356개 렌더링 카드는 고유 리뷰 ID를 갖고 작성자명·평점·작성일도 채워져 있습니다. 16개 행은 작성 본문이 비어 있지만 본문 잘림 표시는 없습니다. 페이지 요약과 8건 차이가 해소되지 않아 모델 학습의 완전한 상호작용 로그로 간주하면 안 됩니다.


In [2]:
from pathlib import Path
import csv
import json
from collections import Counter

CSV_NAME = "diningcode_playwright_national_sungsimdang_20260923.csv.partial"
CHECKPOINT_NAME = "diningcode_playwright_national_sungsimdang_20260923.checkpoint.json"
PROJECT_ROOT = next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "crawler" / "data" / CSV_NAME).exists()
)
CSV_PATH = PROJECT_ROOT / "crawler" / "data" / CSV_NAME
CHECKPOINT_PATH = PROJECT_ROOT / "crawler" / "data" / CHECKPOINT_NAME

with CSV_PATH.open(encoding="utf-8-sig", newline="") as handle:
    rows = list(csv.DictReader(handle))
checkpoint = json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
restaurant_report = next(iter(checkpoint["restaurants"].values()))
print(f"CSV: {CSV_PATH}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Capture status: {restaurant_report['status']}")


CSV: /home/hananthony1/codes/ai-ds-projects/projects/rating_recsys/crawler/data/diningcode_playwright_national_sungsimdang_20260923.csv.partial
Checkpoint: /home/hananthony1/codes/ai-ds-projects/projects/rating_recsys/crawler/data/diningcode_playwright_national_sungsimdang_20260923.checkpoint.json
Capture status: incomplete


In [3]:
review_ids = [row["review_id"].strip() for row in rows if row["review_id"].strip()]
review_id_counts = Counter(review_ids)
expected_count = restaurant_report["expected_review_rows"]
collected_count = len(rows)
metrics = {
    "restaurant": restaurant_report["name"],
    "rows": collected_count,
    "review_more_clicks": restaurant_report["review_more_clicks"],
    "page_evaluation_count": expected_count,
    "unmatched_count": expected_count - collected_count,
    "unmatched_share": round((expected_count - collected_count) / expected_count, 4),
    "unique_review_ids": len(review_id_counts),
    "duplicate_review_ids": sum(count > 1 for count in review_id_counts.values()),
    "missing_user_name": sum(not row["user_name"].strip() for row in rows),
    "missing_user_rating": sum(not row["user_rating"].strip() for row in rows),
    "missing_date": sum(not row["date"].strip() for row in rows),
    "rating_only_empty_body": sum(not row["user_query"].strip() for row in rows),
    "truncated_or_unexpanded_body": sum(row["review_text_complete"] != "true" for row in rows),
    "missing_user_summary_fields": {
        column: sum(not row[column].strip() for row in rows)
        for column in ("user_tot_avg_rating", "user_tot_rating_num", "user_tot_follow_num")
    },
    "optional_review_fields_missing": {
        column: sum(not row[column].strip() for row in rows)
        for column in ("taste", "price", "service", "menu")
    },
    "reason": restaurant_report["reason"],
}
print(json.dumps(metrics, ensure_ascii=False, indent=2))


{
  "restaurant": "성심당 본점",
  "rows": 356,
  "review_more_clicks": 71,
  "page_evaluation_count": 364,
  "unmatched_count": 8,
  "unmatched_share": 0.022,
  "unique_review_ids": 356,
  "duplicate_review_ids": 0,
  "missing_user_name": 0,
  "missing_user_rating": 0,
  "missing_date": 0,
  "rating_only_empty_body": 16,
  "truncated_or_unexpanded_body": 0,
  "missing_user_summary_fields": {
    "user_tot_avg_rating": 0,
    "user_tot_rating_num": 0,
    "user_tot_follow_num": 0
  },
  "optional_review_fields_missing": {
    "taste": 11,
    "price": 11,
    "service": 11,
    "menu": 19
  },
  "reason": "No clickable 평가 더보기 control remains, but the page shows 364 evaluations and only 356 rows were collected."
}
